In [ ]:
# !/usr/bin/env python
# coding: utf-8

import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import plotly.offline as pyo
import sys
from pathlib import Path
module_path = Path("notebooks").resolve()
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
from EDA_functions import *
seed = 5000

# 0. CARGA DE DATOS.

In [ ]:
try:
    df = pickle.load(open(f'../data/processed/df_with_target.sav', 'rb'))
except:
    pkl_path = Path("data/processed") / "df_with_target.sav"
    df = pickle.load(open(pkl_path, 'rb'))


__NOTA:__ Se hará uso de algunas funciones contenidas en el script __EDA_functions.py__.

# 1. TAMAÑO DE DATA - CONTRUCCIONES PRELIMINARES.

Veamos cuál es el tamaño de data.

In [ ]:
df.shape

In [ ]:
df.head(2)

Recordemos la configuración de la variable objetivo `STATUS`.

In [ ]:
print('FRECUENCIA:')
print(df['STATUS'].value_counts())
print('FRECUENCIA RELATIVA:')
print(round((df['STATUS'].value_counts(normalize=True)) * 100), 2)

Se obtiene las variables de __Edad__ y __Años laborando__.

In [ ]:
df["AGE"] = - df['DAYS_BIRTH'] / 365
df["AGE"] = df["AGE"].astype(int)
df['WORK_YEARS'] = -df['DAYS_EMPLOYED'] / 365
df["WORK_YEARS"] = df["WORK_YEARS"].astype(int)

Veamos qué variables poseen valores perdidos y los porcentajes de missing.

In [ ]:
count_missing = get_count_missing(data=df,
                                  feature_names=list(df.columns))
count_missing

La única variable con missing es una variable categórica llamada `OCCUPATION_TYPE`. Se haŕá el debido tratamiento en su momento.

# 2. TRATAMIENTO DE VARIABLES NUMÉRICAS.

## 2.1 Visualización y Control de outliers.

Se identifican las variables numéricas.

In [ ]:
num_names = list(df.select_dtypes(include=['float64', 'int64']).columns)
num_names

Observamos que:
* La 'ID' es una varibale de identificación.
* 'STATUS' es la variable objetivo.
* 'DAYS_BIRTH' y 'DAYS_EMPLOYED' ya han sido tranformadas.
* 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL' son en realidad variables de naturaliza binaria. 

Siendo así no se considerarán como numéricas.

In [ ]:
no_numeric = ['ID',
             'STATUS',
             'DAYS_BIRTH',
             'DAYS_EMPLOYED',
             'FLAG_MOBIL',
             'FLAG_WORK_PHONE',
             'FLAG_PHONE',
             'FLAG_EMAIL']

In [ ]:
num_names = list(set(num_names)-set(no_numeric))
num_names

Al contar con un número no tan grande de variables numéricas, es posible una exploración individual de la distribuciones usando histogramas y box-plots.

In [ ]:
for name in num_names:
    print(f'################################## VARIABLE: {name} ################################################')
    fig = px.histogram(df,
                       x=name,
                       nbins=50,
                       marginal='box')
    fig.show()

Pasa una situación para la variable `WORK_YEARS`. Las personas desempleadas tienen por default un valor de -1000, por lo que para observar la verdadera distribución de la variable, se deben de considerar valores arriba de cero.

In [ ]:
df[df['WORK_YEARS'] <0]['WORK_YEARS'].unique()

In [ ]:
fig = px.histogram(df[df['WORK_YEARS'] >=0],
                   x='WORK_YEARS',
                   nbins=50,
                   marginal='box')
fig.show()

Podemos observar lo siguiente:

* Algunas variables como 'WORK_YEARS' y 'AMT_INCOME_TOTAL' están sesgadas a la derecha. Por ahora no se hará ningúna tranformación para centralizar estas distribuciones.
* Las variables 'AMT_INCOME_TOTAL', 'CNT_FAM_MEMBERS', 'CNT_CHILDREN' cuentan con claros outliers, es decir, puntos que se alejan no solo de la concentración de la información, sino de la cola natural de la distribución. Estos valores son extremos (valores positivos muy grandes).

Para ejercer cierto control de outliers en variables sesgadas a la derecha, se impondrán cotas superiores a las variables involucradas

In [ ]:
df_OUTLIERS = df[(df['AMT_INCOME_TOTAL'] <= 800000)
                 & (df['CNT_FAM_MEMBERS'] <= 8) 
                 & (df['CNT_CHILDREN'] <= 6)]

In [ ]:
df_OUTLIERS

Se comparam las distribuciones de la variable indicada antes y después de outliers.

In [ ]:
skew_num_names = ['AMT_INCOME_TOTAL', 'CNT_FAM_MEMBERS', 'CNT_CHILDREN']
for name in skew_num_names:
    print(f'################################## VARIABLE: {name} ################################################')
    print(f'Antes de outliers.')
    fig = px.histogram(df,
                       x=name,
                       nbins=50,
                       marginal='box')
    fig.show()
    print(f'Después de outliers.')
    fig2 = px.histogram(df_OUTLIERS,
                        x=name,
                        nbins=50,
                        marginal='box')
    fig2.show()

Se comparan los tamaños de data antes y después de outliers.

In [ ]:
print('Porcentaje remanente de la data después de eliminar outliers:', (df_OUTLIERS.shape[0] / df.shape[0]) * 100, '%')

Se puede ver que el porcentaje de data eliminada es ínfimo. Si esto fuera lo contrario, es decir, si data eliminada fuera consideble, se tendría que pensar en otras estrategias para control de outliers y sesgos extremos, como por ejemplo las tranformaciones __yeo-johnson__ o __box-cox__.

## 2.2 Tratamiento de la correlación.

Analicemos la correlación entre  las variables numéricas. Se utilizará la correlación no paramétrica de Spearman.

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df_OUTLIERS[num_names].corr(method='spearman').abs(),
            square=True,
            annot=True,
            cmap='RdBu',
            vmin=-1,
            vmax=1)
plt.show()

La única correlación alta, la de 'CNT_CHILDREN'-'CNT_FAM_MEMBERS' con 82.

In [ ]:
fig = px.scatter(df_OUTLIERS,
                 x='CNT_CHILDREN',
                 y='CNT_FAM_MEMBERS')
fig.show()

Se toma la decisión de eliminar una de estas variables.

In [ ]:
drop_names_by_corr = ['CNT_FAM_MEMBERS']

In [ ]:
df_CORR = df_OUTLIERS.drop(drop_names_by_corr, axis=1)
num_names = [name for name in num_names if name not in drop_names_by_corr]
df_CORR.reset_index(drop=True, inplace=True)

In [ ]:
num_names

## 2.3 Comportamiento con respecto a la variable objetivo.

Veamos el comportamiento de las variables numéricas 'AGE', 'AMT_INCOME_TOTAL', 'CNT_CHILDREN' con respecto al 'STATUS'.

In [ ]:
for name in list(set(num_names)-set(['WORK_YEARS'])):
    print(f'################################## VARIABLE: {name} ################################################')
    fig = px.violin(df_CORR,
                    y=name,
                    x='STATUS',
                    box=True)
    fig.show()
    # fig2 = px.histogram(df_CORR,
    #                     x=name,
    #                     color="STATUS",
    #                     opacity=0.6)
    # fig2.show()

Se observa lo siguiente:
1. Se nota un ligero desface en la distrubución de 'AGE' con respecto a 'STATUS'. Sugiere que los MALOS clientes son uno o dos años más jóvenes.
2. Para 'AMT_INCOME_TOTAL', en el lado de los BUENOS clientes, hay miembros que ganan anualmente más de 500000, algo que no sucede para los MALOS clientes. 
3. Para 'CNT_CHILDREN', hay algunos del lado de los BUENOS clientes que tienen más de 3 hijos y esto no se ve en los MALOS clientes. Hay una mayor concentración en el caso de los BUENOS clientes cercana al cero, por lo que se puede decir que ellos tienden a no tener hijos.

En cuanto a 'WORK_YEARS' se tiene lo siguiente. Se tiene la gráfica para el caso de esta variable para personas *empleadas*.

In [ ]:
fig = px.violin(df_CORR[df_CORR['WORK_YEARS']>=0],
                y='WORK_YEARS',
                x='STATUS',
                box=True)
fig.show()

Es notoria una concentración en los primeros dos años de trabajo para los MALOS clientes. Sería natural pensar que una persona apenas va a entrar a un periodo de estabilización después de recien encontrar trabajo, por ejemplo.

Ahora, para las clientes *desempleados* se tiene lo siguiente.

In [ ]:
df_unemployed = df_CORR[df_CORR['WORK_YEARS'] < 0]
print('% de personas desempleadas con respecto al total:',
      round(((df_unemployed.shape[0]/df_CORR.shape[0]) * 100), 2))

Solo conforman el 15.5% de la data total. El % de morosos que cae en esta "categoría" es el siguiente:

In [ ]:
count_objected = df_unemployed['STATUS'].value_counts().to_frame().reset_index()
count_objected.columns = ['STATUS', 'count']
count_objected['% con respecto al total'] = round((count_objected['count'] / df_CORR.shape[0]) * 100, 2)
count_objected

Con solo el 1.85 % de clientes *desempleados* que caen en mora, se puede ver que esta condición no es tan determinante para ser un cliente "MALO".

In [ ]:
#df_CORR[df_CORR['WORK_YEARS']<0]['STATUS'].value_counts(normalize=True)

## 2.4 Eliminación de variables.

Ya tranformadas, las variables `DAYS_BIRTH`y `DAYS_EMPLOYED` se eliminan.

In [ ]:
unwanted_names = ['DAYS_BIRTH', 'DAYS_EMPLOYED']
df_CORR = df_CORR.drop(unwanted_names, axis=1)
num_names = [name for name in num_names if name not in unwanted_names]

In [ ]:
num_names

# 3. TRATAMIENTO DE VARIABLES CATEGÓRICA.

## 3.1 Cambio de tipo de variables y tratamiento de valores perdidos.

Se determinan las variables categóricas. Para fines prácticos, las variables de naturaleza binaria también 
se considerarán categóricas.

In [ ]:
flag_names = ['FLAG_MOBIL','FLAG_WORK_PHONE','FLAG_PHONE', 'FLAG_EMAIL']
cat_names = list(df_CORR.select_dtypes('object').columns) + flag_names
cat_names

Las variables binarias será tratadas como tipo string como lo son el resto de variables categóricas.

In [ ]:
binary_names = ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'FLAG_MOBIL', 
                'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL']
for name in binary_names:
    df_CORR[name] = df_CORR[name].astype(str)

Se toma la decisión de incluir una categoría que represente los valores perdidos en variables categóricas, más bien en'OCCUPATION_TYPE'. Esta categoría llevará el nombre de "Unidentified".

In [ ]:
df_FILLNA_CATEGORIC = fillna_categoric_data(data=df_CORR,
                                            list_names=cat_names)

## 3.2 Fusión de categorías.

A continuación, se dará un vistazo a los tamaños clase en cada variable categórica, así como a su porcentaje de representación.

In [ ]:
for name in cat_names:
    print(f'################# VARIABLE: {name} #########################################' )
    print('FRECUENCIA:')
    print(df_FILLNA_CATEGORIC[name].value_counts())
    print('FRECUENCIA RELATIVA:')
    print(round((df_FILLNA_CATEGORIC[name].value_counts(normalize=True)) * 100), 2)

Se toma la decisión de fusionar categorías con frecuencia menor o igual a 1%. A continuación, se enlistan las funciones que se encargarán de estas fusiones. Originalmente, este dipo de funciones deben de ir en el script __EDA_fucntions.py__

In [ ]:
def NAME_EDUCATION_TYPE_class(education_type):
    """
    Tiene como tarea homogeneizar los valores de la variable "NAME_EDUCATION_TYPE".
    """
    if education_type in ['Higher education', 'Academic degree']:
        return 'Higher education or Academic degree'
    elif education_type in ['Lower secondary', 'Incomplete higher']:
        return 'Lower secondary or Incomplete higher'
    else:
        return education_type
    

def NAME_HOUSING_TYPE_clas(housing_type):
    """
    Tiene como tarea homogeneizar los valores de la variable "NAME_HOUSING_TYPE".
    """
    if housing_type in ['Rented apartment', 'Office apartment', 'Co-op apartment']:
        return 'Rented apartment or Office apartment or Co-op apartment'
    else:
        return housing_type
        
def OCCUPATION_TYPE_class(occupation_type):
    """
    Tiene como tarea homogeneizar los valores de la variable "OCCUPATION_TYPE".
    """
    if occupation_type in ['Cleaning staff', 'Private service staff', 'Secretaries',
                           'Waiters/barmen staff', 'Low-skill Laborers', 'IT staff',
                           'Realty agents', 'HR staff']:
        return 'Others'
    else:
        return occupation_type

In [ ]:
df_HOMO_CLASS = df_FILLNA_CATEGORIC.copy()

In [ ]:
df_HOMO_CLASS['NAME_EDUCATION_TYPE'] = list(map(NAME_EDUCATION_TYPE_class,
                                                df_HOMO_CLASS['NAME_EDUCATION_TYPE']))
df_HOMO_CLASS['NAME_HOUSING_TYPE'] = list(map(NAME_HOUSING_TYPE_clas,
                                                df_HOMO_CLASS['NAME_HOUSING_TYPE']))
df_HOMO_CLASS['OCCUPATION_TYPE'] = list(map(OCCUPATION_TYPE_class,
                                                df_HOMO_CLASS['OCCUPATION_TYPE']))

Se da un check de las fusiones.

In [ ]:
for name in cat_names:
    print(f'################# VARIABLE: {name} #########################################' )
    print('FRECUENCIA:')
    print(df_HOMO_CLASS[name].value_counts())
    print('FRECUENCIA RELATIVA:')
    print(round((df_HOMO_CLASS[name].value_counts(normalize=True))*100), 2)

Se decide eliminar los 3 únicos miembros en la categoría de estudiantes en 'NAME_INCOME_TYPE'.

In [ ]:
df_HOMO_CLASS = df_HOMO_CLASS[-df_HOMO_CLASS['NAME_INCOME_TYPE'].isin(['Student'])]

In [ ]:
df_HOMO_CLASS.columns

Quedará pendiente desarrollar visualizaciones para poder estudiar como se comportan las variables categóricas, con respecto a la variable `STATUS`.

## 3.4 Eliminación de variables

Se eliminará la variable de género `CODE_GENDER` para prevenir el sesgo del modelo e incurrir en discriminación.
Además, se eliminará la variable `FLAG_MOBIL` porque todos los clientes manifestaron tener teléfono movil.

In [ ]:
unwanted_names = ['CODE_GENDER', 'FLAG_MOBIL']
df_HOMO_CLASS = df_HOMO_CLASS.drop(unwanted_names, axis=1)
cat_names = [name for name in cat_names if name not in unwanted_names]

In [ ]:
df_HOMO_CLASS['STATUS'].value_counts(normalize=True)

# 5. SELECCIÓN DE VARIABLES ANTES DE ENTRENAMIENTO

Aunque este no es el caso, ante un escenario con presencia de muchas variables numéricas o variables categóricas ordinales, uno puede hacer una selección de variables ANTES DE ENTRENAMIENTO usando la __información mutua__, la cual es una métrica traída de la Teoría de la Información. Este es un valor no negativo que mide la dependencia entre variables aleatorias. Valores más altos significan una dependencia más alta.

Análogamente, se puede estudiar la relación de las variables categóricas con el objetivo binario usando *pruebas chi-quadrada de independencia* que pueden aplicarse al hacer uso de tablas de contingencia.

En el caso de las variables numéricas, solo se dará un vistazo de cómo queda la __información mutua__ pero no se usará, en este ejercicio, para selección de variables. En realidad, para nuestro caso, se tienen muy pocas *features*, por lo que se usarán todas y se verá su importancia DESPUÉS DE ENTRENAMIENTO, usando las *features importances* proporcionadas por scikit-learn, o bien usando las *shap feature importances*.

In [ ]:
mutual_information_score(data=df_HOMO_CLASS,
                         feature_names=num_names,
                         y_label_name='STATUS',
                         list_bool_True=None,
                         seed=seed)

Se sugiere que la variable 'AMT_INCOME_TOTAL' es la más importante. Se vará si esto está de acorde a las importancias después de entrenamiento.

# 7. LISTA FINAL DE FEATURES.

Se exihiben qué variables conformarán el conjunto definitivo de features. Están serán puestas en un archivo de configuración config.yaml para un manejo más flexible de algunos scripts, sobre todo en el script de predicción.

In [ ]:
objective_name = 'STATUS'

In [ ]:
num_names

In [ ]:
feature_names = cat_names + num_names
feature_names

# 8. GUARDADO DE LA DATA.

Los conjuntos de datos de __Entrenamiento__ y __Testeo__ (más bien los dataframes) se guardarán en archivos binarios.

In [ ]:
df_HOMO_CLASS

In [ ]:
df_HOMO_CLASS.reset_index(drop=True, inplace=True)

In [ ]:
try:
    pickle.dump(df_HOMO_CLASS, 
                open('../data/processed/df_cleaned_featured.sav', 'wb'))
except:
    pkl_path = Path("data/processed") / "df_cleaned_featured.sav"
    pickle.dump(df_HOMO_CLASS, open(pkl_path, 'wb'))